# Data Validation Pipeline

> **Purpose**: Demonstrate the `validation.py` module — run 6 quality checks and inspect pass/fail results.

**All business logic lives in `../validation.py`. This notebook only imports and calls those functions.**

### Checks performed
| # | Check | What it tests |
|---|-------|---------------|
| 1 | `validate_nulls_and_duplicates` | Missing values per column + fully duplicate rows |
| 2 | `validate_schema` | Expected columns present vs actual |
| 3 | `validate_dtypes` | Column dtype profile (informational) |
| 4 | `validate_dates` | Unparseable dates + future dates |
| 5 | `validate_payment_methods` | Invalid Payment_Method / Transaction_Status values |
| 6 | `validate_regex_patterns` | Transaction_ID must match `^TXN\d+$` |

> **New in this version**: Each violation now includes `percentage` of affected rows, and the summary shows `passed_checks`, `failed_checks`, and `pass_rate`.

> **Previous step**: `cleaning.ipynb` | **Next step**: `rules.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from validation import (
    validate_nulls_and_duplicates,
    validate_schema,
    validate_dtypes,
    validate_dates,
    validate_payment_methods,
    validate_regex_patterns,
    run_validation,
)

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)
print(f"Dataset loaded and cleaned: {df_clean.shape}")

## 1. Null & Duplicate Check

In [ ]:
null_dup = validate_nulls_and_duplicates(df_clean)
print(f"Duplicate rows: {null_dup['duplicate_rows']:,}")
print("\nNull counts per column:")
for col, cnt in null_dup["null_counts"].items():
    if cnt > 0:
        pct = cnt / len(df_clean) * 100
        print(f"  {col:<30}: {cnt:>6,} ({pct:.2f}%)")

## 2. Schema Check

In [ ]:
schema = validate_schema(df_clean)
print(f"Missing columns : {schema['missing_columns']}")
print(f"Extra columns   : {schema['extra_columns']}")

## 3. Data Type Profile

In [ ]:
dtypes = validate_dtypes(df_clean)
pd.DataFrame(dtypes["column_profile"]).T

## 4. Date Validation

In [ ]:
dates = validate_dates(df_clean)
print(f"Invalid (unparseable) dates : {dates['invalid_dates']:,}")
print(f"Future dates                : {dates['future_dates']:,}")

## 5. Payment Method & Status Check

In [ ]:
payment = validate_payment_methods(df_clean)
print(f"Invalid Payment_Method values   : {payment['invalid_payment_methods']:,}")
print(f"Invalid Transaction_Status values: {payment['invalid_statuses']:,}")
for v in payment["violations"]:
    print(f"  [{v['severity']}] {v['rule']} — {v['count']:,} rows ({v['percentage']}%)")
    print(f"    Examples: {v.get('example_values', {})}")

## 6. Regex Pattern Check (Transaction_ID)

In [ ]:
regex = validate_regex_patterns(df_clean)
print(f"Transaction_ID pattern violations: {regex['pattern_violations']:,}")
for v in regex["violations"]:
    print(f"  [{v['severity']}] {v['rule']}")
    print(f"  Count: {v['count']:,} ({v['percentage']}%)")
    print(f"  Example bad values: {v.get('example_values', {})}")

## Full Validation Run & Summary

In [ ]:
val_result = run_validation(df_clean)

print("=== Validation Summary ===")
summary = val_result["summary"]
print(f"  Total checks   : {summary['total_checks']}")
print(f"  Passed checks  : {summary['passed_checks']} ({summary['pass_rate']}%)")
print(f"  Failed checks  : {summary['failed_checks']}")
print(f"  Total violations: {summary['total_violations']}")
print(f"  High severity  : {summary['high_severity']}")
print(f"  Medium severity: {summary['medium_severity']}")
print(f"  Low severity   : {summary['low_severity']}")

print("\n=== All Violations ===")
for v in val_result["violations"]:
    print(f"  [{v['severity']:6}] {v['rule']:<55} {v['count']:>6,} rows ({v['percentage']:.1f}%)")

---
## Key Takeaways

- The `pass_rate` in the summary shows what fraction of checks found zero issues
- Every violation now carries `count` **and** `percentage` — easier to assess impact at a glance
- The `Transaction_ID` regex check flags IDs that don't follow the `TXN<digits>` pattern (likely free-text or UUID formats)
- `Transaction_Status` violations come from values that were not mapped during cleaning (e.g. "Refund", "Complete")
- This result is passed directly to `scoring.py` to compute the `validity_score`